In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.3 Probabilities

A language model does not pick the next character: it produces a
**probability distribution** over every possible next character. Let's
look at one.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless backend (CI has no display)
import matplotlib.pyplot as plt
from pocketlm import train_tiny, pick_config, next_token_probs

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
probs = next_token_probs(model, tok, "ROME")
top = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:10]
plt.bar([repr(c) for c, _ in top], [p for _, p in top])
plt.title("Top-10 next-char probabilities after 'ROME'")
plt.show()

Each bar is the model's belief about the next character. A well-trained
model would put most mass on `'O'`. The whole distribution always sums to 1:

In [ ]:
assert abs(sum(probs.values()) - 1.0) < 1e-5